# SSO / SIMAX Colab Experiment for This Round

First select GPU under "Runtime → Change runtime type". Complete A–D in order and send back the `SSO_RETURN_RESULTS.zip` downloaded at the end. For detailed explanations see `README_FIRST.md` in the package.

This round is not a large-model quality reproduction. Do not automatically run the optional grids in `study.py`. Connecting Drive is recommended to save checkpoints; after a disconnect, upload the same ZIP and keep using the same work directory.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import zipfile, shutil, os, subprocess, sys, json
uploaded = files.upload()  # select SSO_Colab_20260925.zip
choices = [n for n in uploaded if n.endswith('.zip')]
if len(choices) != 1:
    raise ValueError('Please upload only this round\'s single code ZIP')
archive = Path('/content') / choices[0]
archive.write_bytes(uploaded[choices[0]])
code_dir = Path('/content/SSO_Colab_20260925')
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        dest = (Path('/content') / info.filename).resolve()
        if not dest.is_relative_to(code_dir):
            raise ValueError('Archive structure does not match this round\'s code package; please use this round\'s ZIP')
    z.extractall('/content')
os.chdir(code_dir)
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/SSO_SIMAX_20260925')
WORK.mkdir(parents=True, exist_ok=True)
print('Code:', code_dir)
print('Results and checkpoints for this round:', WORK)


## Install dependencies and confirm GPU

Keep Colab's bundled CUDA build of PyTorch; do not install a CPU build of torch. Version info is written into the results. If you are prompted to restart the runtime right after installing, restart and then re-run the upload/connect cell and this cell.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Please select a GPU runtime before continuing.')
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('BF16:', torch.cuda.is_bf16_supported())
(WORK/'environment_pip_freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

def stage(name, dataset='fineweb_edu'):
    subprocess.run([sys.executable, 'run_stage.py', '--stage', name, '--dataset', dataset,
                    '--device', 'cuda', '--work', str(WORK)], check=True)


## A. Correctness and CUDA preflight

Theory path, real forward/backward for the six optimizers, checkpoint resume. On any failure, keep the logs first and do not continue to the training below.

In [ ]:
stage('preflight')


## B. TinyStories 30-step short run on real data

Automatically downloads a small amount of corpus and the tokenizer; runs to step 15 first, then resumes to step 30. This stage validates the pipeline.

In [ ]:
stage('tiny')


## C. FineWeb-Edu real-matrix capture

MuonSphere and SSO-NS each run 100 steps; each trajectory saves three full matrices at steps 1/50/100, 18 in total. The loss here is not used to pick the best method.

In [ ]:
stage('capture', 'fineweb_edu')


## D. Diagnostics on exactly the same snapshots

Runs the PH numerical reference, NS 5/8/12 steps with precision comparison, normal error, and root-finding setting comparison. SVD may be slower than training itself; each snapshot result is saved and can be reused after a restart.

In [ ]:
stage('audit', 'fineweb_edu')


## E. Download and return results

Send this ZIP back as-is. It contains logs, manifests, snapshots, and failure status, but not the raw corpus or checkpoints. Model checkpoints remain in your Drive. Even if a stage fails, you can run this cell on its own to export the existing evidence.

In [ ]:
stage('export')
files.download(str(WORK/'SSO_RETURN_RESULTS.zip'))


## This round ends here

First decide from the results whether to scale up. `study.py` already provides three follow-up plans: repair / precision / ph; see the README for their trigger conditions. The current notebook does not start them.

Do not modify source code / training config and then force-resume an old checkpoint; if you hit an identity mismatch, keep the old directory and logs.